# Expert 01 — Metaclasses
Metaclasses are the 'classes of classes' — they control how classes are created.

## 1. type() — The Default Metaclass

In [ ]:
# type() creates classes dynamically
# type(name, bases, namespace)

# This:
class MyClass:
    x = 10
    def greet(self):
        return f'Hello from {self.__class__.__name__}'

# Is equivalent to:
MyClass2 = type('MyClass2', (), {
    'x': 10,
    'greet': lambda self: f'Hello from {self.__class__.__name__}'
})

obj = MyClass2()
print(obj.greet())
print(type(MyClass))   # <class 'type'>
print(type(type))      # <class 'type'> — type is its own metaclass

## 2. Custom Metaclass

In [ ]:
class SingletonMeta(type):
    """Metaclass that enforces singleton pattern."""
    _instances = {}

    def __call__(cls, *args, **kwargs):
        if cls not in cls._instances:
            cls._instances[cls] = super().__call__(*args, **kwargs)
        return cls._instances[cls]

class Database(metaclass=SingletonMeta):
    def __init__(self, url):
        self.url = url
        print(f'Database created: {url}')

db1 = Database('postgresql://localhost/mydb')
db2 = Database('postgresql://localhost/other')  # ignored
print(db1 is db2)   # True
print(db1.url)      # postgresql://localhost/mydb

## 3. __new__ vs __init__ in Metaclass

In [ ]:
class TrackedMeta(type):
    """Track all classes created with this metaclass."""
    registry = {}

    def __new__(mcs, name, bases, namespace):
        print(f'Creating class: {name}')
        cls = super().__new__(mcs, name, bases, namespace)
        mcs.registry[name] = cls
        return cls

    def __init__(cls, name, bases, namespace):
        super().__init__(name, bases, namespace)
        print(f'Initialized class: {name}')

class Animal(metaclass=TrackedMeta):
    pass

class Dog(Animal):
    pass

class Cat(Animal):
    pass

print('Registry:', list(TrackedMeta.registry.keys()))

## 4. Validation Metaclass

In [ ]:
class ValidatedMeta(type):
    """Enforce that all methods have type annotations."""

    def __new__(mcs, name, bases, namespace):
        for attr_name, attr_val in namespace.items():
            if callable(attr_val) and not attr_name.startswith('_'):
                annotations = getattr(attr_val, '__annotations__', {})
                if 'return' not in annotations:
                    raise TypeError(
                        f'{name}.{attr_name} must have a return type annotation'
                    )
        return super().__new__(mcs, name, bases, namespace)

# This works
class GoodAPI(metaclass=ValidatedMeta):
    def get_user(self, id: int) -> dict:
        return {}

print('GoodAPI created successfully')

# This fails
try:
    class BadAPI(metaclass=ValidatedMeta):
        def get_user(self, id):  # missing return annotation
            return {}
except TypeError as e:
    print(f'Error: {e}')

## 5. __init_subclass__ (Modern Alternative)

In [ ]:
# Python 3.6+ — often replaces metaclasses for subclass hooks
class Plugin:
    _registry = {}

    def __init_subclass__(cls, plugin_name=None, **kwargs):
        super().__init_subclass__(**kwargs)
        name = plugin_name or cls.__name__.lower()
        Plugin._registry[name] = cls
        print(f'Registered plugin: {name}')

class JSONPlugin(Plugin, plugin_name='json'):
    def process(self, data): return str(data)

class CSVPlugin(Plugin, plugin_name='csv'):
    def process(self, data): return ','.join(map(str, data))

print('Registry:', Plugin._registry)
plugin = Plugin._registry['json']()
print(plugin.process({'key': 'value'}))

## 6. Abstract Base Class via Metaclass

In [ ]:
from abc import ABCMeta, abstractmethod

class Shape(metaclass=ABCMeta):
    @abstractmethod
    def area(self) -> float: ...

    @abstractmethod
    def perimeter(self) -> float: ...

    def describe(self) -> str:
        return f'{self.__class__.__name__}: area={self.area():.2f}, perimeter={self.perimeter():.2f}'

class Circle(Shape):
    def __init__(self, r): self.r = r
    def area(self): import math; return math.pi * self.r**2
    def perimeter(self): import math; return 2 * math.pi * self.r

class Rectangle(Shape):
    def __init__(self, w, h): self.w, self.h = w, h
    def area(self): return self.w * self.h
    def perimeter(self): return 2 * (self.w + self.h)

shapes = [Circle(5), Rectangle(4, 6)]
for s in shapes:
    print(s.describe())

# Cannot instantiate abstract class
try:
    Shape()
except TypeError as e:
    print(f'Error: {e}')